# Module 01: Weight Function Comparison

## Learning Objectives
By completing this notebook, you will:
1. Estimate MIDAS models with Beta polynomial, Almon polynomial, and U-MIDAS on the same GDP-IP dataset
2. Compare in-sample and out-of-sample performance across specifications
3. Use AIC/BIC to select among MIDAS variants
4. Visualize how different weight function families fit the data

## Prerequisites
- Notebook 01 (basic MIDAS regression) completed
- Guide 02 (weight functions)
- Guide 03 (U-MIDAS)

## Estimated Time: 15 minutes

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RESOURCES_DIR = '../../module_00_foundations/resources'
if not os.path.exists(RESOURCES_DIR):
    RESOURCES_DIR = '../../../module_00_foundations/resources'

print("Imports complete.")

## 1. Load Data and Build MIDAS Matrix

Same setup as Notebook 01. We use K=9 lags (3 quarterly lags) for this comparison, which gives U-MIDAS a reasonable number of parameters.

In [ ]:
# Load data
gdp = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'gdp_quarterly.csv'),
    index_col='date', parse_dates=True
).squeeze()
gdp.index = pd.PeriodIndex(gdp.index, freq='Q')

ip = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'industrial_production_monthly.csv'),
    index_col='date', parse_dates=True
).squeeze()
ip.index = pd.PeriodIndex(ip.index, freq='M')

# Build MIDAS matrix (reuse from Notebook 01)
def build_midas_matrix(y_q, x_m, n_q_lags=3, m=3):
    K = n_q_lags * m
    T = len(y_q)
    X = np.full((T, K), np.nan)
    q_idx = y_q.index
    m_idx = x_m.index
    
    for t, q in enumerate(q_idx):
        last_m = q.end_time.to_period('M')
        for j in range(K):
            target = last_m - j
            if target in m_idx:
                X[t, j] = x_m.iloc[m_idx.get_loc(target)]
    
    Y = y_q.values
    valid = ~(np.isnan(X).any(axis=1) | np.isnan(Y))
    return Y[valid], X[valid], q_idx[valid]

N_Q_LAGS = 3  # 3 quarterly lags = K=9 monthly lags
Y, X, dates = build_midas_matrix(gdp, ip, n_q_lags=N_Q_LAGS)
K = X.shape[1]
T = len(Y)

print(f"Data: T={T} quarters, K={K} monthly lags")
print(f"Sample: {dates[0]} to {dates[-1]}")

## 2. All Weight Functions

Define all weight function families in one place.

In [ ]:
def beta_weights(K, t1, t2):
    if t1 <= 0 or t2 <= 0:
        return np.ones(K) / K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, t1, t2)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K

def almon_weights(K, t1, t2):
    j = np.arange(K, dtype=float)
    lx = t1 * j + t2 * j**2
    lx -= lx.max()
    raw = np.exp(lx)
    return raw / raw.sum()

def sse_beta(params, Y, X):
    a, b, t1, t2 = params
    if t1 <= 0 or t2 <= 0:
        return 1e10
    w = beta_weights(X.shape[1], t1, t2)
    yh = a + b * (X @ w)
    return np.sum((Y - yh)**2)

def sse_almon(params, Y, X):
    a, b, t1, t2 = params
    w = almon_weights(X.shape[1], t1, t2)
    yh = a + b * (X @ w)
    return np.sum((Y - yh)**2)

def fit_midas(Y, X, sse_fn, p0_list):
    """Fit MIDAS by NLS, trying multiple starting points."""
    best_res, best_sse = None, np.inf
    for p0 in p0_list:
        r = minimize(sse_fn, p0, args=(Y, X), method='Nelder-Mead',
                     options={'maxiter': 15000, 'xatol': 1e-7, 'fatol': 1e-7, 'adaptive': True})
        if r.fun < best_sse:
            best_sse, best_res = r.fun, r
    return best_res

print("Weight functions defined.")

## 3. Estimate All Models

We estimate four models:
1. OLS on equal-weight aggregate (benchmark)
2. Beta MIDAS
3. Almon MIDAS  
4. U-MIDAS (OLS on all K lags)

In [ ]:
# OLS benchmark: equal-weight aggregate
x_agg = X.mean(axis=1)
ols_model = LinearRegression().fit(x_agg.reshape(-1,1), Y)
y_hat_ols = ols_model.predict(x_agg.reshape(-1,1))
r2_ols = r2_score(Y, y_hat_ols)
sse_ols = np.sum((Y - y_hat_ols)**2)

# Beta MIDAS
print("Estimating Beta MIDAS...")
x_eq = X.mean(axis=1)
b_init = np.cov(Y, x_eq)[0,1] / np.var(x_eq)
a_init = Y.mean() - b_init * x_eq.mean()

beta_starts = [
    [a_init, b_init, 1.0, 5.0],
    [a_init, b_init, 1.5, 4.0],
    [a_init, b_init, 1.0, 2.0],
    [a_init, b_init, 2.0, 3.0],
]
beta_res = fit_midas(Y, X, sse_beta, beta_starts)
a_b, b_b, t1_b, t2_b = beta_res.x
t1_b, t2_b = max(t1_b, 0.01), max(t2_b, 0.01)
w_beta = beta_weights(K, t1_b, t2_b)
y_hat_beta = a_b + b_b * (X @ w_beta)
r2_beta = r2_score(Y, y_hat_beta)
sse_beta_val = np.sum((Y - y_hat_beta)**2)
print(f"  Beta MIDAS: R²={r2_beta:.4f}, θ=({t1_b:.3f}, {t2_b:.3f})")

# Almon MIDAS
print("Estimating Almon MIDAS...")
almon_starts = [
    [a_init, b_init, -0.3, 0.0],
    [a_init, b_init, -0.1, -0.02],
    [a_init, b_init, 0.0, 0.0],
    [a_init, b_init, -0.5, 0.0],
]
almon_res = fit_midas(Y, X, sse_almon, almon_starts)
a_a, b_a, t1_a, t2_a = almon_res.x
w_almon = almon_weights(K, t1_a, t2_a)
y_hat_almon = a_a + b_a * (X @ w_almon)
r2_almon = r2_score(Y, y_hat_almon)
sse_almon_val = np.sum((Y - y_hat_almon)**2)
print(f"  Almon MIDAS: R²={r2_almon:.4f}, θ=({t1_a:.3f}, {t2_a:.3f})")

# U-MIDAS (OLS on all lags)
print("Estimating U-MIDAS...")
umidas_model = LinearRegression().fit(X, Y)
y_hat_umidas = umidas_model.predict(X)
r2_umidas = r2_score(Y, y_hat_umidas)
sse_umidas = np.sum((Y - y_hat_umidas)**2)
print(f"  U-MIDAS: R²={r2_umidas:.4f}")

## 4. Model Comparison: AIC and BIC

AIC and BIC penalize for additional parameters. BIC uses $k \ln T$ (stricter than AIC's $2k$).

In [ ]:
def compute_ic(sse, k, n):
    """Compute AIC and BIC from SSE, k parameters, n observations."""
    aic = n * np.log(sse / n) + 2 * k
    bic = n * np.log(sse / n) + k * np.log(n)
    return aic, bic

models = [
    ('Equal-weight OLS', 2, sse_ols, r2_ols),
    ('Beta MIDAS', 4, sse_beta_val, r2_beta),
    ('Almon MIDAS', 4, sse_almon_val, r2_almon),
    (f'U-MIDAS (K={K})', K+1, sse_umidas, r2_umidas),
]

print("Model Comparison Table")
print("=" * 72)
print(f"{'Model':<25} {'Params':>6} {'R²':>8} {'SSE':>10} {'AIC':>10} {'BIC':>10}")
print("-" * 72)

for name, k, sse, r2 in models:
    aic, bic = compute_ic(sse, k, T)
    print(f"{name:<25} {k:>6d} {r2:>8.4f} {sse:>10.3f} {aic:>10.3f} {bic:>10.3f}")

print()
# Identify best models by AIC and BIC
aics = [compute_ic(sse, k, T)[0] for _, k, sse, _ in models]
bics = [compute_ic(sse, k, T)[1] for _, k, sse, _ in models]

best_aic_idx = np.argmin(aics)
best_bic_idx = np.argmin(bics)

print(f"Best by AIC: {models[best_aic_idx][0]}")
print(f"Best by BIC: {models[best_bic_idx][0]}")
print()
print("Note: BIC penalizes more heavily for extra parameters (k*ln(T) vs 2k).")
print(f"For T={T}: BIC penalty per parameter = {np.log(T):.2f} vs AIC = 2.0")

## 5. Weight Function Comparison Plot

The plot shows estimated weights from all three MIDAS variants side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Estimated Weight Functions: Three MIDAS Variants\n'
             f'GDP Growth ~ Monthly IP, K={K} lags, T={T} quarters',
             fontsize=12, fontweight='bold')

lags = np.arange(K)
equal_w = np.ones(K) / K
quarter_boundaries = [3, 6]

# Common plot settings
for ax in axes:
    for qb in quarter_boundaries:
        ax.axvline(qb - 0.5, color='lightgray', linestyle=':', linewidth=1.2)
    ax.axhline(0, color='black', linewidth=0.4)
    ax.set_xlabel('Lag j (0 = most recent month)')
    ax.set_ylabel('Weight')

# Beta MIDAS
ax = axes[0]
ax.bar(lags, w_beta, color='steelblue', alpha=0.8)
ax.plot(lags, equal_w, 'r--', linewidth=1.5, label='Equal weights')
ax.set_title(f'Beta MIDAS\n(θ₁={t1_b:.2f}, θ₂={t2_b:.2f}, R²={r2_beta:.3f})')
ax.legend(fontsize=8)
for i, w in enumerate(w_beta):
    if w > 0.06:
        ax.text(i, w + 0.003, f'{w:.2f}', ha='center', fontsize=7, color='darkblue')

# Almon MIDAS
ax = axes[1]
ax.bar(lags, w_almon, color='coral', alpha=0.8)
ax.plot(lags, equal_w, 'b--', linewidth=1.5, label='Equal weights')
ax.set_title(f'Almon MIDAS\n(θ₁={t1_a:.2f}, θ₂={t2_a:.2f}, R²={r2_almon:.3f})')
ax.legend(fontsize=8)
for i, w in enumerate(w_almon):
    if w > 0.06:
        ax.text(i, w + 0.003, f'{w:.2f}', ha='center', fontsize=7, color='darkred')

# U-MIDAS
ax = axes[2]
phi = umidas_model.coef_
phi_sum = phi.sum()
w_umidas = phi / phi_sum if abs(phi_sum) > 0.01 else np.ones(K)/K

colors = ['steelblue' if w >= 0 else 'coral' for w in w_umidas]
ax.bar(lags, w_umidas, color=colors, alpha=0.8)
ax.plot(lags, equal_w, 'r--', linewidth=1.5, label='Equal weights')
ax.set_title(f'U-MIDAS (OLS)\n(K={K} unrestricted, R²={r2_umidas:.3f})')
ax.legend(fontsize=8)
ax.set_ylim(min(w_umidas.min() - 0.02, -0.05), max(w_umidas.max() + 0.02, 0.15))

# Quarter labels
for ax in axes:
    ax.text(1.0, ax.get_ylim()[1] * 0.95, 'Curr Q', fontsize=7.5, color='gray', ha='center')
    ax.text(4.0, ax.get_ylim()[1] * 0.95, 'Q-1', fontsize=7.5, color='gray', ha='center')
    ax.text(7.0, ax.get_ylim()[1] * 0.95, 'Q-2', fontsize=7.5, color='gray', ha='center')

plt.tight_layout()
plt.savefig('weight_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nObservations:")
print(f"1. Beta and Almon should produce similar shapes (both fit similar decay patterns).")
print(f"2. U-MIDAS may show sign oscillations due to multicollinearity between adjacent monthly lags.")
print(f"3. The overall declining pattern (lag 0 > lag K-1) should be consistent across all three.")

## 6. Out-of-Sample Comparison (Expanding Window)

In-sample R² favors U-MIDAS (more parameters). The true test is out-of-sample. We use an expanding window: train on increasing data, forecast one quarter ahead.

In [ ]:
# Expanding window out-of-sample evaluation
MIN_TRAIN = max(25, K + 10)  # Minimum training observations
n_oos = T - MIN_TRAIN

oos_errors = {'OLS-aggregate': [], 'Beta-MIDAS': [], 'Almon-MIDAS': [], 'U-MIDAS': []}

print(f"Running expanding-window evaluation: {n_oos} out-of-sample periods...")

for end in range(MIN_TRAIN, T):
    Y_train = Y[:end]
    X_train = X[:end]
    Y_test = Y[end]  # One-step-ahead target
    X_test = X[end]
    
    x_agg_train = X_train.mean(axis=1)
    
    # OLS aggregate
    m = LinearRegression().fit(x_agg_train.reshape(-1,1), Y_train)
    yh = m.predict([[X_test.mean()]])[0]
    oos_errors['OLS-aggregate'].append((Y_test - yh)**2)
    
    # Beta MIDAS
    b_i = np.cov(Y_train, x_agg_train)[0,1] / np.var(x_agg_train)
    a_i = Y_train.mean() - b_i * x_agg_train.mean()
    p0_list = [[a_i, b_i, 1.0, 5.0], [a_i, b_i, 1.5, 4.0]]
    res = fit_midas(Y_train, X_train, sse_beta, p0_list)
    a_, b_, t1_, t2_ = res.x
    t1_, t2_ = max(t1_, 0.01), max(t2_, 0.01)
    w_ = beta_weights(K, t1_, t2_)
    yh = a_ + b_ * (X_test @ w_)
    oos_errors['Beta-MIDAS'].append((Y_test - yh)**2)
    
    # Almon MIDAS
    p0_list_a = [[a_i, b_i, -0.3, 0.0], [a_i, b_i, -0.1, -0.02]]
    res_a = fit_midas(Y_train, X_train, sse_almon, p0_list_a)
    a_a_, b_a_, t1_a_, t2_a_ = res_a.x
    w_a_ = almon_weights(K, t1_a_, t2_a_)
    yh_a = a_a_ + b_a_ * (X_test @ w_a_)
    oos_errors['Almon-MIDAS'].append((Y_test - yh_a)**2)
    
    # U-MIDAS
    um = LinearRegression().fit(X_train, Y_train)
    yh_u = um.predict(X_test.reshape(1,-1))[0]
    oos_errors['U-MIDAS'].append((Y_test - yh_u)**2)

# Compute RMSE
print(f"\nOut-of-Sample Results (Expanding Window, {n_oos} periods):")
print(f"{'Model':<20} {'RMSE':>8} {'Relative to OLS':>18}")
print("-" * 50)

ols_rmse = np.sqrt(np.mean(oos_errors['OLS-aggregate']))
for name, errors in oos_errors.items():
    rmse = np.sqrt(np.mean(errors))
    rel = rmse / ols_rmse
    better = "<-- better" if rmse < ols_rmse else ""
    print(f"{name:<20} {rmse:>8.4f} {rel:>14.3f}x {better}")

## 7. Self-Check

Verify your understanding with these automated checks.

In [ ]:
print("Self-Check: Weight Function Comparison")
print("=" * 50)

passed = 0; total = 0

def chk(cond, msg, hint=""):
    global passed, total
    total += 1
    status = "PASS" if cond else "FAIL"
    print(f"  [{status}] {msg}")
    if not cond and hint:
        print(f"         Hint: {hint}")
    if cond: passed += 1

# 1. All weight vectors sum to 1
chk(abs(w_beta.sum() - 1.0) < 1e-8, "Beta weights sum to 1")
chk(abs(w_almon.sum() - 1.0) < 1e-8, "Almon weights sum to 1")

# 2. MIDAS variants have at least as high R² as OLS-aggregate (in sample)
chk(r2_beta >= r2_ols - 0.01,
    f"Beta MIDAS R² ({r2_beta:.4f}) ≥ OLS R² ({r2_ols:.4f})",
    "MIDAS should always fit at least as well as OLS aggregate in sample.")

chk(r2_umidas >= r2_ols - 0.01,
    f"U-MIDAS R² ({r2_umidas:.4f}) ≥ OLS R² ({r2_ols:.4f})",
    "OLS on K lags always fits at least as well as OLS on 1 aggregate.")

# 3. Beta and Almon should produce similar (correlated) weight profiles
weight_corr = np.corrcoef(w_beta, w_almon)[0, 1]
chk(weight_corr > 0.7,
    f"Beta and Almon weight correlation > 0.7: {weight_corr:.3f}",
    "Both families should fit similar shapes — Beta and Almon are often near-equivalent.")

# 4. U-MIDAS has more parameters than Beta MIDAS
chk(K + 1 > 4,
    f"U-MIDAS ({K+1} params) > Beta MIDAS (4 params)",
    "U-MIDAS always has more parameters for K > 3.")

# 5. BIC should prefer more parsimonious model
_, bic_ols = compute_ic(sse_ols, 2, T)
_, bic_beta = compute_ic(sse_beta_val, 4, T)
_, bic_umidas = compute_ic(sse_umidas, K+1, T)

chk(bic_beta < bic_umidas,
    f"BIC favors Beta MIDAS ({bic_beta:.2f}) over U-MIDAS ({bic_umidas:.2f})",
    "BIC's strong penalty for parameters should penalize U-MIDAS more than Beta MIDAS.")

print(f"\nResults: {passed}/{total} checks passed")

## Summary

### What You Compared
- **Beta polynomial MIDAS**: 4 parameters, smooth weight shape, requires NLS
- **Almon polynomial MIDAS**: 4 parameters, unconstrained theta, easier optimization
- **U-MIDAS**: K+1 parameters, no restriction, simple OLS, prone to overfitting

### Key Results
- All three MIDAS variants outperform equal-weight OLS in-sample
- U-MIDAS has highest in-sample R² (more parameters), but not necessarily best OOS
- BIC typically selects Beta or Almon MIDAS over U-MIDAS for K=9 lags
- Beta and Almon weights are highly correlated — both families fit similar shapes

### What's Next
Notebook 03 visualizes the lag polynomial structure interactively, allowing you to explore how changing $\theta$ parameters affects the weight profile and the model fit.

---
*Next: Module 01, Notebook 03 — Lag Polynomial Visualization*